# Poke Agent Training

This notebook runs the training pipeline via the refactored `poke_agent` package.
It replaces the inlined code in `poke_agent_unified.ipynb` with importable modules.

- **Mac / local:** loads existing rollout JSONL and trains with Torch (CUDA, MPS, or CPU).
- **Linux / Kaggle:** can optionally generate a few CABT rollouts inline when `cg-lib` is available.
- **Does not submit** to the competition leaderboard.

Architecture docs: `docs/ARCHITECTURE.md` and `docs/poke-agent-modules.md`.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

# Ensure repo root is importable before loading poke_agent.
ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from poke_agent.paths import print_runtime_info

print_runtime_info(ROOT)
print("torch", torch.__version__)

## 2. Configuration

Defaults come from environment variables (see `docs/ARCHITECTURE.md`).
Override below before building config, or set env vars in the cell.

In [ ]:
# Training requires CABT evaluation rollouts from the cg.game engine by default.
os.environ.setdefault("REQUIRE_CABT_EVAL_DATA", "1")

# Optional overrides — uncomment to change defaults for this session.
# os.environ["PRIMARY_ROLLOUT_DATA"] = "data/mac-rollouts-100k-fullstate.jsonl"
# os.environ["TRAIN_EPOCHS"] = "50"
# os.environ["BATCH_SIZE"] = "128"
# os.environ["MODEL_D_MODEL"] = "512"
# os.environ["CABT_EPISODES"] = "0"  # skip inline rollout generation
# os.environ["REQUIRE_CABT_EVAL_DATA"] = "0"  # smoke test only — allows synthetic fallback

from poke_agent.config import build_config

CONFIG = build_config(ROOT)
CONFIG

## 3. Device and simulator

In [ ]:
from poke_agent.device import torch_device
from poke_agent.simulator import load_simulator, print_simulator_status

DEVICE = torch_device()
print("device", DEVICE)

SIMULATOR = load_simulator(ROOT)
print_simulator_status(SIMULATOR)

## 4. Deck and optional rollout generation

On Mac without `cg-lib`, rollout generation is skipped automatically.
For large datasets, use `scripts/generate_cabt_data.py` or Kaggle/Elmo workers instead.

In [ ]:
from poke_agent.deck import read_deck
from poke_agent.rollout import generate_rollouts

DECK, DECK_SOURCE = read_deck(CONFIG, ROOT)
print("deck cards", len(DECK))
print("deck source", DECK_SOURCE)

GENERATE_EPISODES = int(os.environ.get("CABT_EPISODES", "3" if SIMULATOR.available else "0"))
generate_rollouts(SIMULATOR, DECK, GENERATE_EPISODES, CONFIG["generated_path"])

## 5. Load dataset and build tensors

Picks the first CABT evaluation JSONL from `CONFIG["data_candidates"]`.
Each row must include full `observation` / `action` / `next_observation` payloads
from `scripts/generate_cabt_data.py` (the same `cg.game` engine Kaggle uses).

Training **fails** if no valid CABT evaluation file is found unless
`REQUIRE_CABT_EVAL_DATA=0`.

In [ ]:
from poke_agent.cabt_validation import assert_cabt_evaluation_rows, resolve_cabt_eval_data_path
from poke_agent.dataset import load_jsonl, prepare_training_tensors

DATA_PATH = resolve_cabt_eval_data_path(CONFIG["data_candidates"])
if DATA_PATH is None:
    raise RuntimeError(
        "No CABT evaluation rollout JSONL found. "
        "Run scripts/generate_cabt_data.py or set PRIMARY_ROLLOUT_DATA."
    )

PREVIEW_ROWS = load_jsonl(DATA_PATH)[:5]
assert_cabt_evaluation_rows(PREVIEW_ROWS, path=DATA_PATH, min_rows=1)
print("using CABT evaluation games from", DATA_PATH)

TENSORS = prepare_training_tensors(CONFIG, DEVICE)
print("feature dim", TENSORS.x.shape[1])
print("rows", TENSORS.x.shape[0])

## 6. Build model

In [ ]:
from poke_agent.training import build_model

MODEL = build_model(CONFIG, TENSORS, DEVICE)

## 7. Train

Uses early stopping on total loss. Best weights are restored before checkpoint export.

In [ ]:
from poke_agent.training import train_model

TRAINING_REPORT = train_model(MODEL, TENSORS, CONFIG, DEVICE)
TRAINING_REPORT

## 8. Save checkpoint and report

In [ ]:
from poke_agent.checkpoint import print_training_report, save_checkpoint

OUTPUT_PATH = CONFIG["output_path"]
TRAINING_REPORT = save_checkpoint(
    model=MODEL,
    tensors=TENSORS,
    config=CONFIG,
    training_report=TRAINING_REPORT,
    output_path=OUTPUT_PATH,
)
print_training_report(TRAINING_REPORT, OUTPUT_PATH)

## 9. Inspect checkpoint (optional)

In [ ]:
import torch

checkpoint = torch.load(OUTPUT_PATH, map_location="cpu", weights_only=False)
{
    "model_type": checkpoint["model_type"],
    "input_dim": checkpoint["input_dim"],
    "policy_dim": checkpoint["policy_dim"],
    "model_config": checkpoint["model_config"],
    "best_total_loss": checkpoint["training_report"]["best_total_loss"],
    "best_epoch": checkpoint["training_report"]["best_epoch"],
    "data_path": checkpoint["data_path"],
}